# 2. Databricks: Ossie to Metric View and Back (S3 Backbone)

This notebook reads the Ossie file from S3, builds a Unity Catalog Metric View,
queries it, adds a measure, and exports Ossie back to S3 for the return trip.

The underlying data tables are Snowflake-managed Iceberg tables on the same S3
bucket -- no CSV upload or table creation needed.

Prerequisites:
- External location `ossie-interop-s3` exists pointing to `s3://YOUR-BUCKET-NAME/`
- Storage credential `ossie-s3-credential` is configured
- Notebook 1 has been run (Ossie file exists on S3)

## Step 1 - Install the Apache Ossie Databricks converter

In [ ]:
%pip install "git+https://github.com/apache/ossie.git@01058aa416423cf43a74e7f9fb7f5f70981a418e#subdirectory=converters/databricks"

In [ ]:
dbutils.library.restartPython()

## Step 2 - Configuration

Names match the Snowflake database/schema so Ossie table references work on both sides.

In [ ]:
CATALOG = "demos"
SCHEMA  = "ext_semantic_interop"
SF_NAMESPACE  = "DEMOS.EXT_SEMANTIC_INTEROP"
DBX_NAMESPACE = f"{CATALOG}.{SCHEMA}"
METRIC_VIEW   = f"{CATALOG}.{SCHEMA}.sales_metric_view"

S3_BUCKET = "s3://YOUR-BUCKET-NAME"  # <-- Replace with your S3 bucket
OSSIE_FROM_SF = f"{S3_BUCKET}/ossie/ossie_from_snowflake.yaml"
OSSIE_FROM_DBX = f"{S3_BUCKET}/ossie/ossie_from_databricks.yaml"

print(f"Metric view: {METRIC_VIEW}")
print(f"Reading Ossie from: {OSSIE_FROM_SF}")

## Step 3 - Load Iceberg tables from S3

Snowflake-managed Iceberg tables store data as Parquet with standard Iceberg metadata on S3.
Snowflake appends a random suffix to the base location (e.g. `customers.6EWgT0se/`), so we
discover the actual paths dynamically, read them with `spark.read.format("iceberg")`, and
save as managed tables in Unity Catalog.

### Why not read the Iceberg tables directly?

Snowflake writes standard Apache Iceberg format (format-version 2) to S3 -- the same open format that Databricks claims to support natively. In an ideal world, Databricks would simply `CREATE TABLE ... USING iceberg LOCATION 's3://...'` and read the data in place.

In practice, Databricks' Iceberg support has limitations that prevent this:

- **Managed Iceberg conflict**: Newer Databricks workspaces enable "managed Iceberg" by default, which rejects `CREATE TABLE` with an external `LOCATION` for Iceberg format (`MANAGED_ICEBERG_OPERATION_NOT_SUPPORTED`).
- **Path discovery**: Snowflake appends an opaque suffix to the Iceberg base path (e.g. `customers.6EWgT0se/`), requiring runtime discovery rather than static configuration.

The workaround below reads the Iceberg metadata and Parquet data using `spark.read.format("iceberg")` (which works), then saves it into a Databricks-managed Delta table. This adds a data copy step that would be unnecessary if Databricks supported external Iceberg tables as seamlessly as Snowflake does.

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

# Discover Snowflake-managed Iceberg table paths on S3.
# Snowflake appends a random suffix to the base_location (e.g. customers.6EWgT0se/).
import re

files = dbutils.fs.ls(f"{S3_BUCKET}/iceberg/")
table_paths = {}
for f in files:
    match = re.match(r"(customers|orders)\.\w+/$", f.name)
    if match:
        table_paths[match.group(1)] = f.path.rstrip("/")

print("Discovered Iceberg table paths:", table_paths)

# Read Iceberg tables directly from S3 and register as temp views for querying.
# (Avoids managed-Iceberg conflicts on newer Databricks workspaces.)
for table_name, s3_path in table_paths.items():
    df = spark.read.format("iceberg").load(s3_path)
    df.createOrReplaceTempView(table_name)
    # Also save as permanent tables for the Metric View to reference
    df.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.{table_name}")
    print(f"Loaded {table_name}: {df.count()} rows from {s3_path}")

# Verify data
display(spark.sql(f"""
  SELECT region, SUM(order_amount) total_amount, COUNT(order_id) order_count, SUM(order_qty) total_qty
  FROM {CATALOG}.{SCHEMA}.orders JOIN {CATALOG}.{SCHEMA}.customers USING (customer_id)
  GROUP BY region ORDER BY region"""))
# expect EAST 750/5/12, WEST 700/5/11

## Step 4 - The spec-version and dialect shim

Bridges Snowflake's Ossie 0.1.1 format and the Apache converter's 0.2.0.dev0 format.

In [ ]:
import json
import re
import yaml

CONVERTER_OSSIE_VERSION = "0.2.0.dev0"
SNOWFLAKE_OSSIE_VERSION = "0.1.1"
SNOWFLAKE_DIALECT = "SNOWFLAKE"
ANSI_DIALECT = "ANSI_SQL"
DATABRICKS_DIALECT = "DATABRICKS"


def _relabel_dialects(expression_obj, frm, to):
    if not isinstance(expression_obj, dict):
        return
    for d in expression_obj.get("dialects", []) or []:
        if d.get("dialect") == frm:
            d["dialect"] = to


def snowflake_to_converter(ossie_yaml, drop_fact_fields=True):
    root = yaml.safe_load(ossie_yaml)
    root["version"] = CONVERTER_OSSIE_VERSION
    for model in root.get("semantic_model", []) or []:
        hoisted = []
        for ds in model.get("datasets", []) or []:
            ds_name = ds.get("name", "")
            qual = re.compile(re.escape(ds_name) + r"\.", re.IGNORECASE)
            kept_ext = []
            for ext in ds.get("custom_extensions", []) or []:
                if ext.get("vendor_name") == SNOWFLAKE_DIALECT:
                    blob = json.loads(ext.get("data") or "{}")
                    for m in blob.get("metrics", []) or []:
                        expr = qual.sub("", m["expr"])
                        hoisted.append({"name": m["name"], "expression": {"dialects": [{"dialect": ANSI_DIALECT, "expression": expr}]}})
                else:
                    kept_ext.append(ext)
            if kept_ext:
                ds["custom_extensions"] = kept_ext
            else:
                ds.pop("custom_extensions", None)
            new_fields = []
            for f in ds.get("fields", []) or []:
                _relabel_dialects(f.get("expression"), SNOWFLAKE_DIALECT, ANSI_DIALECT)
                f.pop("custom_extensions", None)
                if drop_fact_fields and "dimension" not in f:
                    continue
                new_fields.append(f)
            if new_fields:
                ds["fields"] = new_fields
            else:
                ds.pop("fields", None)
        if hoisted:
            model["metrics"] = (model.get("metrics", []) or []) + hoisted
    return yaml.safe_dump(root, sort_keys=False)


def converter_to_snowflake(ossie_yaml, dialect=SNOWFLAKE_DIALECT, model_name=None):
    root = yaml.safe_load(ossie_yaml)
    root["version"] = SNOWFLAKE_OSSIE_VERSION
    for model in root.get("semantic_model", []) or []:
        datasets = model.get("datasets", []) or []
        name_map = {}
        for ds in datasets:
            old_name = ds["name"]
            ds["name"] = old_name.upper()
            if old_name != ds["name"]:
                name_map[old_name] = ds["name"]
        for rel in model.get("relationships", []) or []:
            if "from" in rel:
                rel["from"] = rel["from"].upper()
            if "to" in rel:
                rel["to"] = rel["to"].upper()
        fact_ds_name = datasets[0]["name"] if datasets else None
        for ds in datasets:
            is_fact = ds.get("name") == fact_ds_name
            for f in ds.get("fields", []) or []:
                _relabel_dialects(f.get("expression"), DATABRICKS_DIALECT, dialect)
                if not is_fact:
                    f.setdefault("dimension", {})
        fact_cols = []
        ref_re = re.compile(re.escape(fact_ds_name) + r"\.([A-Za-z_]\w*)") if fact_ds_name else None
        for m in model.get("metrics", []) or []:
            _relabel_dialects(m.get("expression"), DATABRICKS_DIALECT, dialect)
            if not fact_ds_name:
                continue
            for d in (m.get("expression") or {}).get("dialects", []) or []:
                if "expression" in d:
                    for old, new in name_map.items():
                        d["expression"] = re.sub(r"\b" + re.escape(old) + r"\.", new + ".", d["expression"])
                    d["expression"] = _qualify_columns(d["expression"], fact_ds_name)
                    for c in ref_re.findall(d["expression"]):
                        if c not in fact_cols:
                            fact_cols.append(c)
        if fact_ds_name and fact_cols:
            fact_ds = datasets[0]
            existing = {f["name"].lower() for f in fact_ds.get("fields", []) or []}
            flds = fact_ds.setdefault("fields", [])
            for c in fact_cols:
                if c.lower() not in existing:
                    flds.append({"name": c.upper(), "expression": {"dialects": [{"dialect": dialect, "expression": c}]}})
        if model_name:
            model["name"] = model_name
    return yaml.safe_dump(root, sort_keys=False)


def _qualify_columns(expr, table):
    return re.sub(r"(?<![\w.])([A-Za-z_]\w*)(?!\s*\()(?![\w.])", lambda m: f"{table}.{m.group(1)}", expr)

## Step 5 - Read Ossie from S3 and convert to Metric View

In [ ]:
from ossie_databricks import convert_ossie_to_metric_view, convert_metric_view_to_ossie

ossie_v1 = dbutils.fs.head(OSSIE_FROM_SF)

# Guard against backslash doubling from cross-platform transfer
try:
    yaml.safe_load(ossie_v1)
except yaml.YAMLError:
    ossie_v1 = ossie_v1.replace('\\\\', '\\\\'[0:1])
    yaml.safe_load(ossie_v1)

print(ossie_v1[:500])

In [ ]:
converter_ready = snowflake_to_converter(ossie_v1)
mv_yaml = convert_ossie_to_metric_view(converter_ready)
mv_yaml = mv_yaml.replace(SF_NAMESPACE, DBX_NAMESPACE)

# Strip unsupported fields (rely) for older Databricks serdes
UNSUPPORTED_JOIN_FIELDS = ("rely",)
def strip_unsupported_fields(mv_yaml_text):
    mv = yaml.safe_load(mv_yaml_text)
    def clean(joins):
        for j in joins or []:
            for field in UNSUPPORTED_JOIN_FIELDS:
                j.pop(field, None)
            clean(j.get("joins"))
    clean(mv.get("joins"))
    return yaml.safe_dump(mv, sort_keys=False)

mv_yaml = strip_unsupported_fields(mv_yaml)
print(mv_yaml)

## Step 6 - Create and query the Metric View

In [ ]:
def create_metric_view(fqname, yaml_body):
    spark.sql('CREATE OR REPLACE VIEW ' + fqname + ' WITH METRICS LANGUAGE YAML AS $$\n' + yaml_body + '\n$$')

create_metric_view(METRIC_VIEW, mv_yaml)
print('created', METRIC_VIEW)

In [ ]:
display(spark.sql(f"""
  SELECT region,
         MEASURE(order_count) AS order_count,
         MEASURE(total_order_amount) AS total_order_amount
  FROM {METRIC_VIEW}
  GROUP BY region ORDER BY region
"""))

## Step 7 - Add a new measure: TOTAL_QUANTITY

This measure is added on the Databricks side and will travel back to Snowflake.

In [ ]:
mv = yaml.safe_load(mv_yaml)
mv.setdefault('measures', []).append({'name': 'TOTAL_QUANTITY', 'expr': 'SUM(order_qty)'})
create_metric_view(METRIC_VIEW, yaml.safe_dump(mv, sort_keys=False))
print('added TOTAL_QUANTITY')

In [ ]:
display(spark.sql(f"""
  SELECT region,
         MEASURE(total_quantity) AS total_quantity,
         MEASURE(total_order_amount) AS total_order_amount,
         MEASURE(order_count) AS order_count
  FROM {METRIC_VIEW}
  GROUP BY region ORDER BY region
"""))
# expect EAST 12/750/5, WEST 11/700/5

## Step 8 - Export updated Ossie back to S3

Read the deployed Metric View YAML, convert back to Snowflake-importable Ossie,
and write directly to S3.

In [ ]:
def get_metric_view_yaml(metric_view_name):
    ddl = spark.sql(f'SHOW CREATE TABLE {metric_view_name}').collect()[0][0]
    start = ddl.index('$') + 2
    end = ddl.index('$', start)
    return ddl[start:end].strip()

mv_yaml_v2 = get_metric_view_yaml(METRIC_VIEW)
ossie_out = convert_metric_view_to_ossie(mv_yaml_v2)
ossie_out = ossie_out.replace(DBX_NAMESPACE, SF_NAMESPACE)
ossie_v2 = converter_to_snowflake(ossie_out, model_name='SALES_SV_V2')

dbutils.fs.put(OSSIE_FROM_DBX, ossie_v2, overwrite=True)
print(ossie_v2)
print(f'\nWritten to {OSSIE_FROM_DBX}')

## Step 9 - Sync Function (for live demo)

To run as a scheduled job: create a Databricks Job with this notebook, schedule every
1 minute. Pause/resume from the Jobs UI during the demo.

In [ ]:
def sync_from_s3():
    """Read latest Ossie from Snowflake, update Metric View, export back."""
    ossie_in = dbutils.fs.head(OSSIE_FROM_SF)
    try:
        yaml.safe_load(ossie_in)
    except yaml.YAMLError:
        ossie_in = ossie_in.replace('\\\\', '\\\\'[0:1])

    cr = snowflake_to_converter(ossie_in)
    mv_body = convert_ossie_to_metric_view(cr)
    mv_body = mv_body.replace(SF_NAMESPACE, DBX_NAMESPACE)
    mv_body = strip_unsupported_fields(mv_body)
    create_metric_view(METRIC_VIEW, mv_body)

    mv_current = get_metric_view_yaml(METRIC_VIEW)
    ossie_out = convert_metric_view_to_ossie(mv_current)
    ossie_out = ossie_out.replace(DBX_NAMESPACE, SF_NAMESPACE)
    ossie_back = converter_to_snowflake(ossie_out, model_name='SALES_SV_V2')
    dbutils.fs.put(OSSIE_FROM_DBX, ossie_back, overwrite=True)
    import datetime
    print(f'Synced at {datetime.datetime.now()}')

# Uncomment to run once:
# sync_from_s3()

## Done

The updated Ossie file (with `TOTAL_QUANTITY`) is at
`s3://YOUR-BUCKET-NAME/ossie/ossie_from_databricks.yaml`.
Open notebook 3 in Snowflake to import it.